# Eksploracja: vector store v2

Sandbox retrievalu na chunkach z `chunker_v2` dla ekstrakcji v2 (`data/extraction_v2/`).
Wybierasz rozdziały i/lub strony → chunking → indexing → query.

- Chunker: `chunker_v2.chunk_document` (parametry: `max_chars` / `overlap_chars`)
- Vector store: `vector_store_v2` (ChromaDB, kolekcja `v2_chunks`)
- Embeddingi: `text-embedding-3-small` (OpenAI API) — wymagany `OPENAI_API_KEY` w `.env`
- Statystyki chunków (`Counter(element_type)`, percentyle długości)
- Sanity-check: brak `picture`, są `table` / `infographic`

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path
from statistics import mean, median

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd

from backend.app.config import V2_TOP_K, VISION_CACHE_DIR
from backend.app.document.models import (
    DocumentMetadata,
    ExtractedChapter,
    ExtractedDocument,
)
from backend.app.document.chunker_v2 import chunk_document
from backend.app.retrieval import vector_store_v2 as vvs

CACHE_DIR = VISION_CACHE_DIR
CHAPTER_ORDER = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
AVAILABLE_CHAPTERS = [cid for cid in CHAPTER_ORDER if (CACHE_DIR / f"{cid}.json").exists()]

print(f"CACHE_DIR:          {CACHE_DIR}")
print(f"Dostępne rozdziały: {AVAILABLE_CHAPTERS}")

CACHE_DIR:          C:\mirror\zadanie\docquery\data\extraction_v2
Dostępne rozdziały: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']


## 2. Wybór zakresu

- `CHAPTERS` — lista ID rozdziałów (np. `["IV", "VI"]`) albo `"ALL"`.
- `PAGE_RANGES` — opcjonalny filtr stron: `"12"`, `"12-13"`, `"12-15,20,25-30"`. `None` = bez filtra.
- `MAX_CHARS`, `OVERLAP_CHARS` — parametry `chunk_document`.

In [2]:
CHAPTERS: list[str] | str = "ALL"
# CHAPTERS = ["IV"]

PAGE_RANGES: str | None = None
# PAGE_RANGES = "12-13"
# PAGE_RANGES = "12-15,20,25-30"

MAX_CHARS = 1500
OVERLAP_CHARS = 200

## 3. Loader + filtr stron

In [3]:
def parse_page_ranges(spec: str | None) -> set[int] | None:
    """'12-15,20,25-30' → {12,13,14,15,20,25,26,27,28,29,30}. None → None."""
    if not spec:
        return None
    out: set[int] = set()
    for part in spec.split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            a, b = part.split("-", 1)
            out.update(range(int(a), int(b) + 1))
        else:
            out.add(int(part))
    return out


def load_document(
    chapters: list[str] | str,
    page_filter: set[int] | None,
) -> ExtractedDocument:
    chapter_ids = AVAILABLE_CHAPTERS if chapters == "ALL" else list(chapters)
    loaded: list[ExtractedChapter] = []
    for ch_id in chapter_ids:
        path = CACHE_DIR / f"{ch_id}.json"
        ch = ExtractedChapter.load_json(path)
        if page_filter is not None:
            ch.pages = [p for p in ch.pages if p.page_num in page_filter]
        if ch.pages:
            loaded.append(ch)
    if not loaded:
        raise ValueError("Brak stron po filtrze — sprawdź CHAPTERS/PAGE_RANGES.")
    total_pages = sum(len(ch.pages) for ch in loaded)
    return ExtractedDocument(
        metadata=DocumentMetadata(
            source_file="raport_2024_pl.pdf",
            total_pages=total_pages,
            extraction_date="notebook",
        ),
        title="BGK 2024 v2 (podzbiór)",
        chapters=loaded,
    )


page_filter = parse_page_ranges(PAGE_RANGES)
doc = load_document(CHAPTERS, page_filter)

block_types = Counter(b.element_type for b in doc.get_all_blocks())
print(f"Rozdziały:      {[ch.chapter_id for ch in doc.chapters]}")
print(f"Łącznie stron:  {len(doc.get_all_pages())}")
print(f"Łącznie bloków: {sum(block_types.values())}")
print("Typy bloków:")
for t, n in block_types.most_common():
    print(f"  {t:<20} {n}")

Rozdziały:      ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
Łącznie stron:  182
Łącznie bloków: 2242
Typy bloków:
  text                 1071
  subsection-header    362
  list                 209
  identifier           133
  table                108
  picture              105
  caption              94
  footnote             59
  section-header       57
  infographic          44


## 4. Chunking + podgląd

In [4]:
chunks = chunk_document(doc, max_chars=MAX_CHARS, overlap_chars=OVERLAP_CHARS)

chunk_types = Counter(c["element_type"] for c in chunks)
lengths = [len(c["content"]) for c in chunks]

print(f"Liczba chunków: {len(chunks)}")
print("Typy chunków:")
for t, n in chunk_types.most_common():
    print(f"  {t:<12} {n}")

if lengths:
    print(
        "\nDługość `content` (znaki): "
        f"min={min(lengths)}  median={int(median(lengths))}  "
        f"mean={int(mean(lengths))}  max={max(lengths)}"
    )

assert "picture" not in chunk_types, "picture powinien być pomijany (SKIP)"
print("\nOK: brak chunków typu `picture` (oczekiwane — SKIP).")

Liczba chunków: 521
Typy chunków:
  text         369
  table        108
  infographic  44

Długość `content` (znaki): min=53  median=1238  mean=1129  max=3731

OK: brak chunków typu `picture` (oczekiwane — SKIP).


In [5]:
df = pd.DataFrame(
    [
        {
            "idx": c["chunk_index"],
            "chapter": c["chapter"],
            "section": c["section"],
            "type": c["element_type"],
            "pages": c["pages"],
            "len": len(c["content"]),
            "search_preview": c["search_text"][:200].replace("\n", " "),
        }
        for c in chunks
    ]
)
df.head(20)

,idx,chapter,section,type,pages,len,search_preview
0,0,I Jedyny taki bank,1. List Prezesa Zarządu,text,[4],1233,I Jedyny taki bank > 1. List Prezesa Zarządu ...
1,1,I Jedyny taki bank,1. List Prezesa Zarządu,text,[4],1346,I Jedyny taki bank > 1. List Prezesa Zarządu ...
2,2,I Jedyny taki bank,1. List Prezesa Zarządu,text,[4],901,I Jedyny taki bank > 1. List Prezesa Zarządu ...
3,3,I Jedyny taki bank,1. List Prezesa Zarządu,text,"[4, 5]",1370,I Jedyny taki bank > 1. List Prezesa Zarządu S...
4,4,I Jedyny taki bank,1. List Prezesa Zarządu,text,[5],1373,I Jedyny taki bank > 1. List Prezesa Zarządu ...
5,5,I Jedyny taki bank,1. List Prezesa Zarządu,text,[5],690,I Jedyny taki bank > 1. List Prezesa Zarządu ...
6,6,I Jedyny taki bank,2. Kim jesteśmy?,text,[6],783,I Jedyny taki bank > 2. Kim jesteśmy? GRI 2-1...
7,7,I Jedyny taki bank,3. Podsumowanie 2024 roku,infographic,[6],282,I Jedyny taki bank > 3. Podsumowanie 2024 roku...
8,8,I Jedyny taki bank,3. Podsumowanie 2024 roku,text,"[6, 7]",1382,I Jedyny taki bank > 3. Podsumowanie 2024 roku...
9,9,I Jedyny taki bank,3. Podsumowanie 2024 roku,infographic,[7],840,I Jedyny taki bank > 3. Podsumowanie 2024 roku...


## 5. Indexing

`reset_collection()` czyści kolekcję z poprzednich runów, żeby eksperymenty się nie nakładały.

In [6]:
vvs.reset_collection()
vvs.index_v2_chunks(chunks)
print(f"Zaindeksowano {len(chunks)} chunków w kolekcji 'v2_chunks'.")

Zaindeksowano 521 chunków w kolekcji 'v2_chunks'.


## 6. Query sandbox

In [7]:
def show(query: str, top_k: int = V2_TOP_K, max_preview: int = 2000) -> None:
    results = vvs.search_v2(query, top_k=top_k)
    print(f"QUERY: {query}\n")
    for i, r in enumerate(results, 1):
        header = f"{r['chapter'] or '-'} › {r['section'] or '-'}"
        print(
            f"[{i}] {header}  strony={r['pages']}  "
            f"type={r['element_type']}  dist={r['distance']:.3f}"
        )
        print(r["content"][:max_preview].strip())
        if len(r["content"]) > max_preview:
            print("...")
        print()

In [8]:
show("W którym roku zostało uruchomione Forum Klienta?")

QUERY: W którym roku zostało uruchomione Forum Klienta?

[1] IV Otoczenie › 3. Doświadczenia klientów  strony=[51]  type=text  dist=0.469
Przygotowujemy rekomendacje zmian, przekazujemy je do właściwych jednostek banku i planujemy inicjatywy proklienckie.

W 2024 roku uruchomiliśmy Forum Klienta. Są to spotkania, podczas których analizujemy z właścicielami biznesowymi główne punkty niezadowolenia naszych klientów, przygotowujemy rekomendacje oraz sposób i harmonogram wprowadzania zmian.

Budujemy kulturę klientocentryczną

Chcemy, aby pracownicy byli świadomi swojego wpływu na doświadczenia klientów i nieustannie rozwijamy kulturę organizacyjną, w której klienci są w centrum uwagi. Doceniamy klientocentryczne postawy pracowników. Jeśli klienci pozytywnie wyrażają się o współpracy z daną osobą, przekazujemy jej informacje na ten temat.

Prowadzimy też cykl warsztatów edukacyjnych z pracownikami. Wiedzę o głosie klienta udostępniamy w intranetowej witrynie CX.

Zarządzamy relacjami

Rozw

In [9]:
show("W jaki sposób klienci mogą składać reklamacje?")

QUERY: W jaki sposób klienci mogą składać reklamacje?

[1] IV Otoczenie › 3. Doświadczenia klientów  strony=[51, 52]  type=text  dist=0.304
Te relacje są równie ważnym elementem naszej działalności.

Odpowiadamy na reklamacje

Głos naszych klientów analizujemy także na podstawie skarg i reklamacji. Ponad 80% respondentów uważa, że nasze odpowiedzi na reklamacje są jasne i czytelne. Piszemy je zgodnie z zasadami prostego języka.

Cały czas pracujemy też nad skróceniem czasu potrzebnego do udzielenia odpowiedzi. Przygotowaliśmy m.in. szablony dla powtarzalnych zgłoszeń. Rejestracja reklamacji odbywa się w aplikacji, w której gromadzimy całą dokumentację.

Łącznie w 2024 r. do banku wpłynęły 333 skargi i reklamacje, z czego 35% uznaliśmy za zasadne. Wskaźnik odwołań od decyzji reklamacyjnych wyniósł zaś 5%.

Przyjmujemy reklamacje i skargi klientów banku, a także klientów naszych pośredników kasowych, finansowych i partnerów finansujących. 50% reklamacji innych niż reklamacje usług płatni

In [10]:
show("Jakie zmiany w składzie zarządu nastąpiły w 2024 r.?")

QUERY: Jakie zmiany w składzie zarządu nastąpiły w 2024 r.?

[1] VI Ład korporacyjny › 4. Zasady, zadania i zakres działania organów i komitetów  strony=[112]  type=table  dist=0.285
| Tabela 63. Skład Zarządu BGK, funkcje oraz frekwencja w posiedzeniach w 2024 r.

Skład Zarządu BGK, funkcje oraz frekwencja w posiedzeniach w 2024 r.
Tabela zawiera dane osobowe członków zarządu, ich funkcje, okresy pełnienia, daty powołania, daty upływu kadencji oraz frekwencję w posiedzeniach.

| imię i nazwisko | funkcja | okres pełnienia funkcji w 2024 r. | data powołania do Zarządu | data upływu obecnej kadencji | frekwencja w posiedzeniach w 2024 r. (obecność na posiedzeniach/ liczba posiedzeń w trakcie sprawowania mandatu) |
|---|---|---|---|---|---|
| Mirosław Czekaj | p.o. prezesa / prezes | 12.04 – 17.07.2024 / 18.07– 31.12.2024 | 12.04.2024 | 24.09.2026 | 42/42 |
| Marta Postuła | pierwsza wiceprezes | 12.04 – 31.12.2024 | 12.04.2024 | 24.09.2026 | 41/42 |
| Maciej Kliś | wiceprezes | 26.04 – 

In [11]:
# własne eksperymenty:
# show("...")

## 7. Uwaga

Notebook używa `chunker_v2` + `vector_store_v2` (embeddingi: `text-embedding-3-small` przez OpenAI API).
Wymagany `OPENAI_API_KEY` w `.env`.